In [1]:

#THIS IS A COPY OF 5.4_simple.py to check volpy settings
# version 5 integrates new correlation map, also to add help add width/height filtering and cell grid allignments
#this is v5.py with updated volpy fit changes in 5.3 but without multitrial registration components
#version 4 of test_single_trial_RAM_DISK.py with updated MATLAB .mat saving (sped up)
# TO RUN: conda activate caiman
# # python C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\test_single_trial_RAM_DISK_5.4_simple.py C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B

froot = r'C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B'
import argparse
import os
import re

# parser = argparse.ArgumentParser()
# parser.add_argument("froot", help="Path to the input movie file")
# args = parser.parse_args()

# froot = args.froot
#find all folders in r'C:\caiman_data\testdata\testdata\NF107.6B' and make list of those folder paths     

folder_paths = []
base_path = froot
for root, dirs, files in os.walk(base_path):
    for dir_name in dirs:
        folder_paths.append(os.path.join(root, dir_name))

# regex for folders like FOV1_T1, FOV12_T3, etc.
pattern = re.compile(r"^FOV\d+_T\d+$")

matching_folders = []

for folder in folder_paths:
    for item in os.listdir(folder):
        item_path = os.path.join(folder, item)
        if os.path.isdir(item_path) and pattern.match(item):
            matching_folders.append(item_path)

print(matching_folders)

from collections import defaultdict
import os
import re

# group folders by FOV number
fov_groups = defaultdict(list)

for path in matching_folders:
    folder_name = os.path.basename(path)
    match = re.match(r"^FOV(\d+)_T(\d+)$", folder_name)
    if match:
        fov_number = match.group(1)  # e.g. "1" from FOV1_T2
        fov_groups[fov_number].append(path)

# sort each FOV group by date and T number
for fov in fov_groups:
    fov_groups[fov].sort(
        key=lambda p: (
            int(os.path.basename(os.path.dirname(p))),  # date: 20250505
            int(re.search(r"_T(\d+)$", os.path.basename(p)).group(1))  # trial number
        )
    )

for fov, paths in sorted(fov_groups.items(), key=lambda x: int(x[0])):
    print(f"Analyzing FOV{fov} with {len(paths)} sessions")



['C:\\Users\\ICNLab\\caiman_data\\testdata\\testdata\\NF107.6B\\20250505\\FOV1_T1', 'C:\\Users\\ICNLab\\caiman_data\\testdata\\testdata\\NF107.6B\\20250505\\FOV1_T2', 'C:\\Users\\ICNLab\\caiman_data\\testdata\\testdata\\NF107.6B\\20250519\\FOV1_T1']
Analyzing FOV1 with 3 sessions


In [2]:

print("Importing packages and Initializing...")
import matplotlib
matplotlib.use("Qtagg")   # non-interactive, no windows
print(matplotlib.get_backend())
from base64 import b64encode
import cv2
import glob
import h5py
import imageio
from IPython import get_ipython
from IPython.display import HTML, display, clear_output
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from pathlib import Path
from PIL import Image
import re

#import to cover extras from single_trial.py
import gc
import scipy.io
from scipy import stats
from scipy.signal import butter, lfilter
from scipy.signal import savgol_filter
import sys
import mat73
import pandas as pd


from pathlib import Path

try:
    cv2.setNumThreads(0)
except:
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
        get_ipython().run_line_magic('matplotlib', 'qt')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.utils.utils import download_demo, download_model
from caiman.source_extraction.volpy import utils
from caiman.source_extraction.volpy.volparams import volparams
from caiman.source_extraction.volpy.volpy import VOLPY
from caiman.source_extraction.volpy.mrcnn import visualize, neurons
import caiman.source_extraction.volpy.mrcnn.model as modellib
from caiman.summary_images import local_correlations_movie_offline
from caiman.summary_images import mean_image
from caiman.paths import caiman_datadir
from caiman.summary_images import local_correlations_movie_in_memory
import gc


logging.basicConfig(format=
                    "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s]" \
                    "[%(process)d] %(message)s",
                    level=logging.ERROR)

print(paths[0])

folder_path = paths[0]

Importing packages and Initializing...
Qtagg
C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1


In [4]:


# find the .tsm file in the folder
tsm_files = [f for f in os.listdir(folder_path) if f.endswith('.tsm')]
fname = os.path.join(folder_path, tsm_files[0])
print("Processing file:", fname)



##
#fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM2\FOV1_T2.tsm'
fr = 640
print(fname, fr)


##
# Cleanup R:/ drive
print("Cleaning up R:/ drive...")
def safe_close_mmap(arr):
    try:
        if hasattr(arr, "base") and hasattr(arr.base, "close"):
            arr.base.close()
    except Exception as e:
        print("close failed:", e)


# 1. Delete any Python references to memmaps pointing to R:/
try:
    safe_close_mmap(Yr)  # or whatever your memmap object is called
except NameError:
    pass

try:
    safe_close_mmap(mmap_file_rig)  # or whatever your memmap object is called
except NameError:
    pass

gc.collect()  # force Python to release the memory mapping

# 2. Delete all files in R:/
for f in Path(r'R:/').glob('*'):
    if f.is_file():
        f.unlink()
print("Cleared all files from R:/")


##
pw_rigid = False  # flag for pw-rigid motion correction
gsig_filt = (3, 3)  # size of filter, in general gSig (see below),
# change this one if algorithm does not work
max_shifts = (5, 5)  # maximum allowed rigid shift
strides = (48, 48)  # start a new patch for pw-rigid motion correction every x pixels
overlaps = (24, 24)  # overlap between paths (size of patch strides+overlaps)
max_deviation_rigid = 3  # maximum deviation allowed for patch with respect to rigid shifts
border_nan = 'copy'
use_cuda = True

opts_dict = {
    'fnames': fname,
    'fr': fr,
    'pw_rigid': pw_rigid,
    'max_shifts': max_shifts,
    'gSig_filt': gsig_filt,
    'strides': strides,
    'overlaps': overlaps,
    'max_deviation_rigid': max_deviation_rigid,
    'border_nan': border_nan,
    'use_cuda': use_cuda
}

opts = volparams(params_dict=opts_dict)

##
print("Loading data...")
m_orig = cm.load(fname)
ds_ratio = 0.2

##
c, dview, n_processes = cm.cluster.setup_cluster(
            backend='local', n_processes=None, single_thread=False)

##
print("Motion correction...")
mc = MotionCorrect(fname, dview=dview, **opts.get_group('motion'))
mc.motion_correct(save_movie=True)
#about 2.3 minutes for 12800 frames (2m 13-21 s)
print("Done.")

##
print("Loading corrected movie...")
m_rig = cm.load(mc.mmap_file) # 11s
ds_ratio = 0.2
print("Done.")

del m_orig
gc.collect()

##CONVERT ORDER FROM C TO F VIA DATA STREAMING
name = Path(mc.mmap_file[0]).name

Y = int(re.search(r'_d1_(\d+)', name).group(1))
X = int(re.search(r'_d2_(\d+)', name).group(1))
T = int(re.search(r'_frames_(\d+)', name).group(1))

shape = (T, Y, X)


src = np.memmap(
    mc.mmap_file[0],
    dtype='float32',
    mode='r+',
    shape=(T, Y, X),
    order='C'   # matches physical layout
)

print("Saving stabilized movie to RAM-disk...")
# Path to RAM disk memmap
p = Path(fname)
ram_path = Path(r'R:/') / f"{p.stem}_rig__d1_{m_rig.shape[1]}_d2_{m_rig.shape[2]}_d3_1_order_C_frames_{m_rig.shape[0]}.mmap"
ram_path = str(ram_path).replace("/", "\\") 

dst = np.memmap(
    ram_path,
    dtype='float32',
    mode='w+',
    shape=(T, Y, X),
    order='F'   # true pixel-wise contiguous time
)

chunk = 16  # tune this

for t0 in range(0, T, chunk):
    t1 = min(t0 + chunk, T)

    block = src[t0:t1]      # small buffer
    dst[t0:t1] = block     # repacked to F order

    # optional but recommended on RAM disk
    #src[t0:t1] = 0.0       # free backing pages
    
dst.flush()

if hasattr(src, 'base') and hasattr(src.base, 'close'):
    src.base.close()
    
if hasattr(dst, 'base') and hasattr(dst.base, 'close'):
    dst.base.close()

del src, dst
gc.collect()





##

print("Computing mean and correlation images...")
img = np.mean(m_rig, axis=0)
img = (img-np.mean(img))/np.std(img)


import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from tqdm import tqdm

# ===============================
# 1. Parameters
# ===============================
SHAPE = (12800, 512, 512)
BANDPASS = (5, 300) # Hz #70,300


# ===============================
# 2. Load memory-mapped video
# ===============================

video = np.memmap(
    mc.mmap_file[0],
    dtype=np.float32,
    mode="r",
    shape=SHAPE,
    order="C"
).swapaxes(1, 2)

T, H, W = video.shape
print(f"Loaded video: {video.shape}")

# ===============================
# 3. High-pass filter (bandpass-compatible API)
# ===============================
def bandpass_filter(data, fs, low, high=None, order=3):
    """
    High-pass filter using the 'low' cutoff.
    The 'high' argument is accepted for API compatibility but ignored.
    """
    nyq = 0.5 * fs
    b, a = butter(order, low / nyq, btype="high")
    return filtfilt(b, a, data, axis=0)

# ===============================
# Parameters
# ===============================
TILE_SIZE = 4
H, W = 512, 512
FRAME_RATE = fr
# (low, high), high ignored
DISPLAY_CLIP = 99

# ===============================
# Coherence metric
# ===============================
def coherence_metric(tile_filt):
    """
    tile_filt: shape (T, Npix)
    Returns mean pixel-to-tile correlation.
    """
    # Tile reference (subthreshold signals sum coherently)
    ref = tile_filt.mean(axis=1)


    ref -= ref.mean()
    ref_std = ref.std() + 1e-9

    # Normalize reference
    ref /= ref_std

    # Normalize pixels
    pix = tile_filt - tile_filt.mean(axis=0)
    pix /= (pix.std(axis=0) + 1e-9)

    # Correlation with reference
    corr = np.mean(ref[:, None] * pix, axis=0)

    # Use mean absolute correlation as coherence
    return np.mean(np.abs(corr))


# ===============================
# Output tile map
# ===============================
n_tiles_y = H // TILE_SIZE
n_tiles_x = W // TILE_SIZE

tile_coherence_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)

# ===============================
# Main loop
# ===============================
with tqdm(total=n_tiles_y * n_tiles_x, desc="Computing coherence") as pbar:
    for ty in range(n_tiles_y):
        for tx in range(n_tiles_x):

            y0 = ty * TILE_SIZE
            y1 = y0 + TILE_SIZE
            x0 = tx * TILE_SIZE
            x1 = x0 + TILE_SIZE

            # Extract tile: (T, 16, 16)
            tile = video[:, y0:y1, x0:x1]
            tile = tile.reshape(T, -1)

            # High-pass filter all pixels independently
            tile_filt = bandpass_filter(
                tile, FRAME_RATE, *BANDPASS
            )

            # Compute coherence
            tile_coherence_map[ty, tx] = coherence_metric(tile_filt)

            pbar.update(1)

# ===============================
# Expand to image resolution
# ===============================
coherence_image = np.repeat(
    np.repeat(tile_coherence_map, TILE_SIZE, axis=0),
    TILE_SIZE, axis=1
)

if hasattr(video, 'base') and hasattr(video.base, 'close'):
    video.base.close()

del video
gc.collect()

# ===============================
# Visualization
# ===============================
vmax = np.percentile(coherence_image, DISPLAY_CLIP)

plt.figure(figsize=(6, 6))
plt.imshow(coherence_image, cmap="viridis", vmin=0, vmax=vmax)
plt.title("Grid-based subthreshold coherence ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
plt.colorbar(label="Mean |pixel–tile correlation|")
plt.axis("off")
plt.tight_layout()
plt.show()
plt.close('all')

img_corr = coherence_image
summary_images = np.stack([img, img, img_corr], axis=0).astype(np.float32)
cm.movie(summary_images).save(fname[:-5]+'_summary_images.tif')

plt.imshow(summary_images[0], cmap='gray')
plt.axis('off')
plt.savefig(fname[:-4]+'_mean.tif', format='tif', bbox_inches='tight', pad_inches=0)
plt.close('all') # Save the figure and close the plot   

plt.imshow(summary_images[2], cmap='gray')
plt.axis('off')
plt.savefig(fname[:-4]+'_corr.tif', format='tif', bbox_inches='tight', pad_inches=0)
plt.close('all') # Save the figure and close the plot   
img = summary_images.transpose([1, 2, 0])


print(fname[:-4]+'_corr.tif')
height, width = img.shape[:2]
print(img.shape)

# --------------------------------------------------------------
# Extract channels like MATLAB
# --------------------------------------------------------------
R = img[:, :, 0]
B = img[:, :, 2]

# --------------------------------------------------------------
# MATLAB-style normalization (mat2gray + uint8)
# --------------------------------------------------------------
def normalize_like_matlab(x):
    x = x.astype(np.float64)
    mn = x.min()
    mx = x.max()
    x = (x - mn) / (mx - mn + 1e-12)

    # MATLAB uint8 applies rounding, not floor
    x = np.round(255 * x).astype(np.uint8)
    return x

R_norm = normalize_like_matlab(R)
B_norm = normalize_like_matlab(B)

# --------------------------------------------------------------
# Build MATLAB-equivalent RGB (R,R,B)
# --------------------------------------------------------------
rgb = np.stack([R_norm, R_norm, B_norm], axis=2).astype(np.uint8)

# --------------------------------------------------------------
# Save as PNG (MATLAB-compatible pixel data)
# --------------------------------------------------------------
outname = fname[:-4] + "_py.png"
Image.fromarray(rgb).save(outname)

print("Saved:", outname)
img = rgb.copy()



##
print("Running Mask R-CNN inference...")
weights_path="C:/Users/ICNLab/caiman_data/testdata/testdata/mask_rcnn_neuron_0012.h5"
#download_model('mask_rcnn')
#ROIs, r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=True)
r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=False)
ROIs = r['masks'].transpose([2, 0, 1])
Coords = r['rois']
cm.movie(ROIs).save(fname[:-4]+'newmrcnn_ROIs.hdf5')

fig, axs = plt.subplots(1, 2)
axs[0].imshow(summary_images[1])
axs[1].imshow(ROIs.sum(0))
axs[0].set_title('mean image')
axs[1].set_title('masks')
plt.savefig(fname[:-6] + 'newmrcnn_ROIs.png', format='png', bbox_inches='tight', pad_inches=0)
plt.close('all')# Save the figure and close the plot   

#save ROIs as npy array
np.save(fname[:-4]+'newmrcnn_ROIs.npy', ROIs)
print("Saved ROIs as npy array:", fname[:-4]+'newmrcnn_ROIs.npy')

###NEW SECTION FOR ROI COORDINATE EXTRACTION
cell_centers = [((y1 + y2) // 2, (x1 + x2) // 2) for (y1, x1, y2, x2) in Coords]
cell_centers = np.array(cell_centers)
print("Cell centers:", cell_centers)    
#display the cell centers on the image
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img, cmap='gray') # Display the image
ax.scatter(cell_centers[:, 1], cell_centers[:, 0], color='red') # Display the cell centers
ax.set_title('Cell centers')    # Set the title of the plot
plt.savefig(fname[:-4] + '_cell_centers.png', format='png', bbox_inches='tight', pad_inches=0)
plt.close('all') # Save the figure and close the plot     

# Save to a file
save_path = fname[:-4] + '_cell_centers.npy'
np.save(save_path, cell_centers)

print(f"Cell centers saved to {save_path}")

cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False, maxtasksperchild=1)


Processing file: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1\FOV1_T1.tsm
C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1\FOV1_T1.tsm 640
Cleaning up R:/ drive...
Cleared all files from R:/
Loading data...
Motion correction...
Saving mmap to:  R:/FOV1_T1_rig__d1_512_d2_512_d3_1_order_F_frames_12800.mmap
Done.
Loading corrected movie...


100%|██████████| 1/1 [00:09<00:00,  9.68s/it]


Done.
Saving stabilized movie to RAM-disk...
Computing mean and correlation images...
Loaded video: (12800, 512, 512)


Computing coherence: 100%|██████████| 16384/16384 [02:01<00:00, 134.47it/s]


C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1\FOV1_T1_corr.tif
(512, 512, 3)
Saved: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1\FOV1_T1_py.png
Running Mask R-CNN inference...

Configurations:
BACKBONE                       resnet50
BACKBONE_STRIDES               [4, 8, 16, 32, 64]
BATCH_SIZE                     1
BBOX_STD_DEV                   [0.1 0.1 0.2 0.2]
COMPUTE_BACKBONE_SHAPE         None
DETECTION_MAX_INSTANCES        200
DETECTION_MIN_CONFIDENCE       0
DETECTION_NMS_THRESHOLD        0.3
FPN_CLASSIF_FC_LAYERS_SIZE     1024
GPU_COUNT                      1
GRADIENT_CLIP_NORM             5.0
IMAGES_PER_GPU                 1
IMAGE_CHANNEL_COUNT            3
IMAGE_MAX_DIM                  512
IMAGE_META_SIZE                14
IMAGE_MIN_DIM                  512
IMAGE_MIN_SCALE                0
IMAGE_RESIZE_MODE              crop
IMAGE_SHAPE                    [512 512   3]
LEARNING_MOMENTUM              0.9
LEARNING_RATE       

     1846431 [deprecation.py:            new_func():554][43528] From c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\tensorflow\python\util\deprecation.py:629: calling map_fn_v2 (from tensorflow.python.ops.map_fn) with dtype is deprecated and will be removed in a future version.
Instructions for updating:
Use fn_output_signature instead


Processing 1 images
image                    shape: (512, 512, 3)         min:    0.00000  max:  255.00000  uint8
molded_images            shape: (1, 512, 512, 3)      min:  -91.11000  max:  168.24000  float64
image_metas              shape: (1, 14)               min:    0.00000  max:  512.00000  int32
anchors                  shape: (1, 65472, 4)         min:   -0.04428  max:    1.01297  float32


c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


MADE FIGURE
Saved ROIs as npy array: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1\FOV1_T1newmrcnn_ROIs.npy
Cell centers: [[278 136]
 [235 208]
 [369  90]
 [462 434]
 [313 262]
 [446 364]
 [ 96 500]
 [172  66]
 [ 55 180]
 [138 432]
 [164 319]
 [ 26 353]
 [ 64 364]
 [260 466]
 [ 97 380]
 [212  43]
 [195 132]
 [462 418]
 [373 293]
 [223  27]
 [211 104]
 [163 290]
 [430 100]
 [174 266]
 [300  84]
 [350 293]
 [ 91 408]
 [ 99 138]
 [329 408]
 [130 114]
 [183 343]
 [428  76]
 [496 455]
 [101 264]
 [327 288]
 [ 63 429]
 [ 90 227]
 [128 482]
 [ 57 247]
 [261 151]
 [148  55]
 [343 150]
 [138 346]
 [156 353]
 [106 428]
 [501  29]
 [157 193]
 [145 274]
 [117 185]
 [204 243]
 [159 332]
 [161 163]
 [368 251]
 [133 168]
 [149 404]
 [125 206]
 [305 124]
 [241 405]
 [141 470]
 [316  19]
 [164 478]
 [249 119]
 [335 190]
 [428 383]
 [148 222]
 [171 375]
 [274 346]
 [369 190]
 [277 211]
 [157 391]
 [447 245]
 [ 97 308]
 [154 247]
 [180 149]
 [293 155]
 [230 448]
 [274 328]
 [280

In [5]:
cm.stop_server(dview=dview)


In [17]:

##
pw_rigid = False  # flag for pw-rigid motion correction
gsig_filt = (3, 3)  # size of filter, in general gSig (see below),
# change this one if algorithm does not work
max_shifts = (5, 5)  # maximum allowed rigid shift
strides = (48, 48)  # start a new patch for pw-rigid motion correction every x pixels
overlaps = (24, 24)  # overlap between paths (size of patch strides+overlaps)
max_deviation_rigid = 3  # maximum deviation allowed for patch with respect to rigid shifts
border_nan = 'copy'
use_cuda = True

opts_dict = {
    'fnames': fname,
    'fr': fr,
    'pw_rigid': pw_rigid,
    'max_shifts': max_shifts,
    'gSig_filt': gsig_filt,
    'strides': strides,
    'overlaps': overlaps,
    'max_deviation_rigid': max_deviation_rigid,
    'border_nan': border_nan,
    'use_cuda': use_cuda
}

opts = volparams(params_dict=opts_dict)


In [ ]:
##
c, dview, n_processes = cm.cluster.setup_cluster(
            backend='local', n_processes=None, single_thread=False)
##
ROIs = ROIs                                   # region of interests
index = list(range(len(ROIs)))                # index of neurons
weights = None                                # if None, use ROIs for initialization; to reuse weights check reuse weights block

template_size = 0.008                         # half size of the window length for spike templates, default is 20 ms
context_size = 35                             # number of pixels surrounding the ROI to censor from the background PCA
visualize_ROI = False                         # whether to visualize the region of interest inside the context region
hp_freq_pb = 1 / 3                            # parameter for high-pass filter to remove photobleaching
clip = 100                                    # maximum number of spikes to form spike template
threshold_method = 'simple'  #adaptive_threshold               # adaptive_threshold or simple
min_spikes= 10                                # minimal spikes to be found
pnorm = 0.5                                   # a variable deciding the amount of spikes chosen for adaptive threshold method
threshold = 4         #5                        # threshold for finding spikes only used in simple threshold method, Increase the threshold to find less spikes
do_plot = False                               # plot detail of spikes, template for the last iteration
ridge_bg= 0.05                                # ridge regression regularizer strength for background removement, larger value specifies stronger regularization
sub_freq = 20                                 # frequency for subthreshold extraction
weight_update = 'ridge'                       # ridge or NMF for weight update
n_iter = 2                                    # number of iterations alternating between estimating spike times and spatial filters
censor_size = 5                               # size of the censoring region around the ROI
min_width = 0                                 #minumum half peak-height width in ms
max_width = 9                                 #maximum half peak-height width in ms      
w_h_ratio = 1                                 #minumum ratio of height in %dF/F over half peak-height width in ms
#hp_freq = 30                                  #high-pass frequency to apply before spike detection in denoise spikes function

bob = 'volpy_thresh4_newfilt9max_wh0'
correl_cutoff = 1
snr_thresh_display = 0.00001


opts_dict={'fnames': ram_path,   #'fnames': fname_new,
        'ROIs': ROIs,
        'index': index,
        'weights': weights,
        'template_size': template_size,
        'context_size': context_size,
        'visualize_ROI': visualize_ROI,
        'hp_freq_pb': hp_freq_pb,
        'clip': clip,
        'threshold_method': threshold_method,
        'min_spikes':min_spikes,
        'pnorm': pnorm,
        'threshold': threshold,
        'do_plot':do_plot,
        'ridge_bg':ridge_bg,
        'sub_freq': sub_freq,
        'weight_update': weight_update,
        'n_iter': n_iter,
        'censor_size': censor_size,
        'min_width': min_width,
        'max_width': max_width,
        'w_h_ratio': w_h_ratio
        #'hp_freq': hp_freq
        }

opts.change_params(params_dict=opts_dict)
#opts = volparams(params_dict=opts_dict)

vpy = VOLPY(n_processes=n_processes, dview=dview, params=opts)

print("Running VOLPY fit...")
vpy.fit(n_processes=n_processes, dview=dview)
#takes a while to run
print("Done.")

# Visualize spatial footprints and traces
#print(np.where(vpy.estimates['locality'])[0])    # neurons that pass locality test
# idx = np.where(vpy.estimates['locality'] > 0)[0]
# utils.view_components(vpy.estimates, img_corr, idx)


##

# Reconstructed movie
# flip_signal = True    
# mv_all = utils.reconstructed_movie(vpy.estimates.copy(), fnames=mc.mmap_file,
#                                         idx=idx, scope=(0,1000), flip_signal=flip_signal)
#mv_all.play(fr=40, magnification=3)

##
vpy.estimates['ROIs'] = ROIs
vpy.estimates['Coords'] = Coords
save_name = fname[:-4]+'new_volpy'
np.save(save_name, vpy.estimates)

cm.stop_server(dview=dview)
log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)

print("Saved VOLPY estimates to:", save_name + '.npy')


print(vpy.estimates.keys())
print(len(vpy.estimates['spikes']))
#print(len(vpy.estimates['spikeTimes']))
print(vpy.estimates['snr']) 

#print length of each key's data:
for key in vpy.estimates.keys():
    print(f"{key}: {len(vpy.estimates[key])}")

#print number of neurons with snr > 3
snr_threshold = 3.0 #########################################################################################################
high_snr_neurons = np.sum(vpy.estimates['snr'] > snr_threshold)
print(f"Number of neurons with SNR > {snr_threshold}: {high_snr_neurons}")


##
vpynew = vpy.estimates
#vpynew['spikes'] = np.array(vpynew['spikes'], dtype=object)

try:
    num_frames = np.max(vpynew['dFF'].shape)
    dur = num_frames/640
    vpynew['snr_over_thresh'] = []

    vpynew['raster'] = np.zeros_like(vpynew['dFF'])
    vpynew['firing_rate'] = np.zeros_like(vpynew['dFF'])
    vpynew['unique_trace'] = []
    vpynew['cell_idxs'] = []

    for i in range(vpynew['dFF'].shape[0]-1):
        vpynew['raster'][i, vpynew['spikes'][i]] = 1
        vpynew['firing_rate'][i] = savgol_filter(np.convolve(vpynew['raster'][i]*640,np.ones(32)/32,mode='same'),64,1)

    for i in range(len(vpynew['ROIs'])):
        vpynew['snr_over_thresh'].append(vpynew['snr'][i] >= snr_thresh_display) #################################################################################################################
    print("SNR LIST", vpynew['snr'])
    print("snr_over_thresh", vpynew['snr_over_thresh'])
    print("Number of neurons with SNR > 0:", np.sum(vpynew['snr_over_thresh']))

    if np.sum(vpynew['snr_over_thresh']) > 0:
        to_remove = set()
        dFF = np.array(vpynew['dFF']).astype(float)
        R = np.corrcoef(dFF)
        idx0, idx1 = np.where(np.triu(R, 1) > correl_cutoff) #################################################################################################################
        max_vals = np.max(dFF, axis=1)
        smaller = np.where(max_vals[idx0] < max_vals[idx1], idx0, idx1)
        to_remove.update(smaller.tolist())
        vpynew['unique_trace'] = [True if x not in to_remove else False for x in range(len(vpynew['ROIs']))]

    print(vpynew['unique_trace'])
    print("Correl cutoff", correl_cutoff)
    print("There are", np.sum(vpynew['unique_trace']), "unique traces after correlation filtering.")
    print("And there were ", len(to_remove), "traces removed due to high correlation.")

    vpynew['cell_idxs'] = []
    for cell in range(len(vpynew['ROIs'])):
        if vpynew['snr_over_thresh'][cell] and vpynew['unique_trace'][cell]:
            vpynew['cell_idxs'].append(cell)

    print("Final number of cells after SNR and correlation filtering:", len(vpynew['cell_idxs']))
    print(vpynew['cell_idxs'])
    print(len(vpynew['cell_idxs']))

    #make figure

    cells = np.array(vpynew['cell_idxs'])
    time = np.arange(0,dur,1/640)

    fig = plt.figure(figsize=(8.0, 11.0), facecolor='w',constrained_layout=True)
    spec = fig.add_gridspec(ncols=3, nrows=5, width_ratios=[1,1,1], height_ratios=[2, 5,1,1,1])
    ax1 = fig.add_subplot(spec[0, 0])
    ax2 = fig.add_subplot(spec[0, 1])
    ax_text = fig.add_subplot(spec[0, 2],facecolor='w')
    ax3 = fig.add_subplot(spec[1, :],facecolor='w')
    ax4 = fig.add_subplot(spec[4, :],facecolor='w')
    ax5 = fig.add_subplot(spec[2, :],facecolor='w')
    ax5r = ax5.twinx()
    ax6 = fig.add_subplot(spec[3, :],facecolor='w')
    #ax7 = fig.add_subplot(spec[4, :],facecolor='w')

    ax1.imshow(img[:,:,1], cmap='gray')
    ax2.imshow(img[:,:,2], cmap='gray')
    ax1.set_title('Mean image',color='k',fontsize=14)
    ax2.set_title('Corr image',color='k',fontsize=14)
    ax1.set_axis_off()
    ax2.set_axis_off()
    ax_text.set_axis_off()

    llim = 0
    if len(cells)>0:
        pos_cells = []
        neg_cells = []
        b, a = butter(1, [1.5, 100], fs=640, btype='band')
        k = 1
        for i in range(0, len(cells)):
            if ''.join(vpynew['polarity'][cells[i]]) in 'negative':
                color = '#9AAB3A'
                mult = -1
                neg_cells.append(cells[i])
            else:
                color = '#54A0A8'
                mult = 1
                pos_cells.append(cells[i])
            y = np.array(lfilter(b,a,stats.zscore(np.array(vpynew['dFF'][cells[i]] * mult * 100,dtype=np.float32))) + ((k - 1) * 8)).reshape(1,num_frames)
            ax3.plot(llim+time,y[0,:],color, linewidth=0.3)
            ax3.plot(llim+time[vpynew['spikes'][cells[i]]],np.max(y)*np.ones(vpynew['spikes'][cells[i]].shape[0]),"|",color='firebrick',markersize=2)
            k = k + 1


        if len(pos_cells)>0:
            mean_fr_pos = np.mean(vpynew['firing_rate'][pos_cells,:], axis=0)
            sem_pos = stats.sem(np.array(vpynew['firing_rate'][pos_cells,:],dtype=np.float32), axis=0)
            ax5r.plot(llim+time, np.array(mean_fr_pos,dtype='float32').ravel(), label='Mean firing rate', color='#54A0A8',linewidth=0.3)
            ax5r.fill_between(llim+time, np.array(mean_fr_pos - sem_pos,dtype='float32').ravel(), np.array(mean_fr_pos + sem_pos,dtype='float32'), color='#54A0A8', alpha=0.3, label='SEM')
            ax5.set_ylabel('Firing rate (Hz)',color='#54A0A8',fontsize=12)
            ax5r.tick_params(axis ='y', labelcolor = '#54A0A8')
        if len(neg_cells)>0:
            mean_fr_neg = np.mean(vpynew['firing_rate'][neg_cells,:], axis=0)
            sem_neg = stats.sem(np.array(vpynew['firing_rate'][neg_cells,:],dtype=np.float32), axis=0)
            ax5.plot(llim+time, np.array(mean_fr_neg,dtype='float32').ravel(), label='Mean firing rate', color='#9AAB3A',linewidth=0.3)
            ax5.fill_between(llim+time, np.array(mean_fr_neg - sem_neg,dtype='float32').ravel(), np.array(mean_fr_neg + sem_neg,dtype='float32'), color='#9AAB3A', alpha=0.3, label='SEM')
            ax5.set_ylabel('Firing rate (Hz)',color='#9AAB3A',fontsize=12)
            ax5r.tick_params(axis ='y', labelcolor = '#9AAB3A')

    wheel_mat = os.path.dirname(fname) + '\\Wheel.mat'
    if os.path.exists(wheel_mat):
        wheel=mat73.loadmat(wheel_mat)
        if 'behavior' in wheel:
            ax4.plot(wheel['behavior'][:,0],wheel['behavior'][:,1],'r',linewidth=1.2)
            if wheel['behavior'].shape[1]>2:
                ax4.plot(wheel['behavior'][:,0],wheel['behavior'][:,2],'k',linewidth=1)
            ax4.set_ylabel('Behavior',color='k',fontsize=12)
            ax4.set_yticks([-1,0,1])
            ax4.set_ylim([-2,2])

        if wheel['data_time'].any():
            whl_time = np.arange(0,np.max(wheel['data_time']),1/640)
            wheel_interp = np.interp(whl_time, wheel['data_time'], wheel['data_pos'])
            speed = np.zeros_like(wheel_interp)
            for i in range(0,len(whl_time)-1):
                speed[i] = (wheel_interp[i+1]-wheel_interp[i])/(whl_time[i+1]-whl_time[i])

            #speed[speed>100] = 0
            #speed[speed<0] = 0
            speed = savgol_filter(speed,64,1)
            ax6.plot(whl_time,speed,'k',linewidth=1)
            ax6.set_ylabel('Speed (cm/s)',color='k',fontsize=12)
            #ax7.plot(whl_time,speed,'w',linewidth=1)
            #ax7.set_ylabel('Speed (cm/s)',color='k',fontsize=12)

        ax_text.text(0.5, 0.8, 'Mouse = ' + wheel['mouse'], color='k',fontsize=10, ha='center')
        ax_text.text(0.5, 0.4, 'Stimulus = ' + wheel['stimulus'], color='k',fontsize=10, ha='center')
        ax_text.text(0.5, 0.6, 'Date = ' + str(np.array(wheel['currentdate'],dtype='int32')), color='k',fontsize=10, ha='center')
        #ax_text.text(0.5, 0.2, 'File = ' + wheel['file'], color='w',fontsize=10, ha='center')
        if wheel['stimulus']=='Map' and 'rand_num' in wheel:
            ax_text.text(0.5, 0, 'Field = ' + " ".join(str(x) for x in wheel['rand_num'].astype(int)), color='k',fontsize=10, ha='center')
        elif wheel['stimulus']=='Tuning' and 'rand_num' in wheel:
            ax_text.text(0.5, 0, 'Orientation = ' + " ".join(str(x) for x in wheel['rand_num'].astype(int)), color='k',fontsize=10, ha='center')
        elif wheel['stimulus']=='Tuning' and 'rand_num' in wheel:
            ax_text.text(0.5, 0, 'Orientation = ' + " ".join(str(x) for x in wheel['rand_num'].astype(int)), color='k',fontsize=10, ha='center')
    else:
        print("Wheel data does not exist")


    for ax in [ax3,ax4,ax5,ax6]:
        ax.tick_params(color='black', labelcolor='black')
        ax.set_xlabel('Time (sec)',color='k',fontsize=12)
        ax.set_xlim([llim,llim+dur])
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
    ax3.set_title('dFF',color='k',fontsize=14)
    ax3.set_ylabel(r'$\Delta$F/F (%)',color='k',fontsize=12)


    fig.savefig(fname[:-4] + '_'+ bob +'.pdf')
    #plt.close('all')
    
    print("Saved VOLPY figure to:", fname[:-4] + '_volpy.pdf')

    print("Saving VOLPY data to MAT file...")
    vpynew['ROIs'] = ROIs
    #vpy['rect'] = r['rois']
    vpynew['img'] = img
    del vpynew['rawROI']
    #scipy.io.savemat(fname[:-4] + '_volpy.mat', {'vpynew': vpynew}, format='5', do_compression=True)





    print("Converting data types for fast saving...")

    # Keys identified from inspection output that need fixing
    keys_to_convert_float = [
        't', 'ts', 't_rec', 't_sub', 'templates', 'snr', 
        'thresh', 'weights', 'locality', 'context_coord', 'F0', 'dFF', 
        'raster', 'firing_rate'
    ]

    keys_to_convert_int = [
        'num_spikes'
    ]

    # Process float conversions
    for key in keys_to_convert_float:
        if key in vpynew and vpynew[key].dtype == object:
            try:
                # Attempt a direct conversion to float32 (fastest for scientific data)
                vpynew[key] = np.array(vpynew[key], dtype=np.float32)
                print(f"  Converted '{key}' to float32 array.")
            except ValueError:
                print(f"  Could not convert '{key}' to standard array dtype. Keeping as object array.")

    # Process integer conversions
    for key in keys_to_convert_int:
        if key in vpynew and vpynew[key].dtype == object:
            try:
                vpynew[key] = np.array(vpynew[key], dtype=np.int32)
                print(f"  Converted '{key}' to int32 array.")
            except ValueError:
                print(f"  Could not convert '{key}' to int32 array. Keeping as object array.")

    # Handle variables that are inherently irregular lists that MUST be object arrays in Python, 
    # but we ensure they are clean for saving.

    # Handle 'mean_im', 'cell_n', 'polarity' (irregular shapes/strings)
    for key in ['mean_im', 'cell_n', 'polarity']:
        if key in vpynew and vpynew[key].dtype == object:
            vpynew[key] = np.array(vpynew[key], dtype=object) # Ensure they are formally object arrays

    # Handle spikes and low_spikes. The try/except handles the 'bool is not iterable' error.
    if vpynew['spikes'].dtype == object:
        vpynew['spikes'] = np.array([list(x) for x in vpynew['spikes']], dtype=object)
        
    if vpynew['low_spikes'].dtype == object:
        try:
            # This was causing the TypeError because it was actually a boolean array
            vpynew['low_spikes'] = np.array([list(x) for x in vpynew['low_spikes']], dtype=object)
        except TypeError:
            # If it's a bool array, just make sure it's saved as a clean boolean array
            vpynew['low_spikes'] = np.array(vpynew['low_spikes'], dtype=bool) 


    print("Data type conversion complete.")

    scipy.io.savemat(fname[:-4] + '_volpy.mat', {'vpynew': vpynew}, format='5', do_compression=True)
    print("Saved VOLPY data to:", fname[:-4] + '_volpy.mat')


    # vpynew.estimates['params'] = opts
    # save_name = f'volpy_{os.path.split(fnames)[1][:-5]}_{threshold_method}'
    # np.save(fnames[:-4] + '_volpy.npy', vpynew.estimates)

    #del vpynew
    # % STOP CLUSTER and clean up log files

    log_files = glob.glob('*_LOG_*')
    for log_file in log_files:
        os.remove(log_file)
except ValueError:
    traceback.print_exc()
    print("No volpy data was saved")

Running VOLPY fit...
Starting VOLPY spike detection...
Done.
Saved VOLPY estimates to: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1\FOV1_T1new_volpy.npy
dict_keys(['rawROI', 'mean_im', 'cell_n', 't', 'ts', 't_rec', 't_sub', 'spikes', 'low_spikes', 'num_spikes', 'templates', 'snr', 'thresh', 'weights', 'locality', 'context_coord', 'F0', 'dFF', 'polarity', 'ROIs', 'Coords'])
96
[8.85931197247681 5.905357387036641 6.420370926788471 -6.9934196898464975
 0 5.164379239771911 0 5.646088679802719 7.492996135704115
 3.044711055156413 -5.1702036274601015 -5.289194911164546
 4.425685236999113 -5.321497837744917 0 5.359200291763735
 5.582720204968527 4.563729205095881 5.710047164477416 -4.19781668919652
 -5.724447785674873 0 3.043319460048159 5.603323007446349
 3.1052178914168955 5.253290668100503 4.089843902019025 -4.72644044710947
 -6.723683487713356 4.860053678635675 5.447500963752387 0.6105922739126868
 -4.9802131800482545 0.250955399657779 4.889932253990359 0
 4.647